In [11]:
from langchain_groq import ChatGroq


In [ ]:
llm = ChatGroq(
    groq_api_key = 'gsk_######################',
    model="llama-3.3-70b-versatile",
    temperature=0.6
)

In [28]:
response = llm.invoke(  
    "The first person to land on moon?"
)
print(response.content)

The first person to land on the moon was Neil Armstrong. He stepped out of the lunar module Eagle and onto the moon's surface on July 20, 1969, during the Apollo 11 mission. Armstrong famously declared, "That's one small step for man, one giant leap for mankind," as he became the first human to set foot on the moon.


In [15]:
from langchain_community.document_loaders import WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


install bs4 ----> It is used to parse and extract readable content from HTML webpages.

In [71]:
loader = WebBaseLoader("https://www.amazon.jobs/en/jobs/10418971/machine-learning-engineer-ii-eu-intech-exports-emerging-and-expansions")
page_data = loader.load().pop().page_content
print(page_data)

Machine Learning Engineer II, EU INTech - Exports, Emerging and Expansions - Job ID: 10418971 | Amazon.jobs
Skip to main content×HomeTeamsLocationsJob categoriesMy careerMy applicationsMy profileAccount securitySettingsSign outResourcesAccommodationsBenefitsInclusive experiencesHow We HireLeadership principlesWorking at AmazonFAQMachine Learning Engineer II, EU INTech - Exports, Emerging and ExpansionsJob ID: 10418971 | ADCI - KarnatakaApply nowDescriptionAre you obsessed with solving challenging problems? Do you have exceptional software engineering skills? Are you interested into Search Engine Technologies, Document Retrieval and Machine Learning? Are you constantly looking for ways to improve your skills, your software, and your organization?Amazon is looking for a passionate, talented, and inventive Machine Learning Engineer to join our E3 team. You will work with the largest online retail search application in the world, both in terms of users, catalogue size, and computing resour

* now we will write the prompt to extract the main data only, for that we import 

In [72]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
    """
    ###   SCARPED TEXT FORM WEBSITE :
    {page_data}
    ### INSTRUCTION:
    The scraped text is form  the career's page of a website.
    Your role is to extract the job postings and return them in JSON format contaning the following roles : 'role', 'experience', 'skills' and 'description'. 
    ### VALID JSON (NO PREAMBLE):                                   
    """
)

chain_extract = prompt_extract | llm
res = chain_extract.invoke(input = {'page_data': page_data })
print(res.content)


```json
{
  "role": "Machine Learning Engineer II",
  "experience": "2+ years of non-internship professional software development experience",
  "skills": [
    "Data structure implementation",
    "Basic algorithm development",
    "Object-oriented design principles",
    "Big Data",
    "NoSQL",
    "Map-Reduce",
    "Machine learning approaches and algorithms",
    "Highly concurrent, high throughput systems",
    "Complex distributed systems",
    "Programming languages (C++, Java, or related)",
    "Scripting languages (Perl, Python, or equivalent)",
    "Document Retrieval (indexing)",
    "Search Engine Technologies (e.g. Lucene)",
    "Document Understanding (Natural Language Processing)"
  ],
  "description": "Amazon is looking for a passionate, talented, and inventive Machine Learning Engineer to join our E3 team. You will work with the largest online retail search application in the world, both in terms of users, catalogue size, and computing resources, and your work will di

In [73]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

{'role': 'Machine Learning Engineer II',
 'experience': '2+ years of non-internship professional software development experience',
 'skills': ['Data structure implementation',
  'Basic algorithm development',
  'Object-oriented design principles',
  'Big Data',
  'NoSQL',
  'Map-Reduce',
  'Machine learning approaches and algorithms',
  'Highly concurrent, high throughput systems',
  'Complex distributed systems',
  'Programming languages (C++, Java, or related)',
  'Scripting languages (Perl, Python, or equivalent)',
  'Document Retrieval (indexing)',
  'Search Engine Technologies (e.g. Lucene)',
  'Document Understanding (Natural Language Processing)'],
 'description': 'Amazon is looking for a passionate, talented, and inventive Machine Learning Engineer to join our E3 team. You will work with the largest online retail search application in the world, both in terms of users, catalogue size, and computing resources, and your work will directly impact millions of customers.'}

In [74]:
import pandas as pd

df = pd.read_csv('My_Skill_Stack.csv')
df

,Techstack,Projects
0,"Java,",https://github.com/Aryanmishra299/Guessing-the...
1,"Java, OOPS",https://github.com/Aryanmishra299/Online-libra...
2,Python,https://github.com/Aryanmishra299/Snake-Water-...
3,"Data Analysis, SQL, Excel, Data Visualization",https://github.com/Aryanmishra299/Pizza_Sales_SQL
4,"Python, OOPS, Pandas, Numpy, Seaborn, Data Vis...",https://github.com/Aryanmishra299/Customer-Chu...


In [75]:
import chromadb
import uuid

client =   chromadb.PersistentClient('VectorStore')
collection = client.get_or_create_collection(name="JOB_Applying")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents = row['Techstack'],
                metadatas = {'links': row['Projects']},
                ids = [str(uuid.uuid4())])


In [79]:
job = json_res
job['skills']

['Data structure implementation',
 'Basic algorithm development',
 'Object-oriented design principles',
 'Big Data',
 'NoSQL',
 'Map-Reduce',
 'Machine learning approaches and algorithms',
 'Highly concurrent, high throughput systems',
 'Complex distributed systems',
 'Programming languages (C++, Java, or related)',
 'Scripting languages (Perl, Python, or equivalent)',
 'Document Retrieval (indexing)',
 'Search Engine Technologies (e.g. Lucene)',
 'Document Understanding (Natural Language Processing)']

In [84]:
links = collection.query(query_texts= job['skills'], n_results =1).get('metadatas',[])
links

[[{'links': 'https://github.com/Aryanmishra299/Pizza_Sales_SQL'}],
 [{'links': 'https://github.com/Aryanmishra299/Snake-Water-Gun-game'}],
 [{'links': 'https://github.com/Aryanmishra299/Guessing-the-number-'}],
 [{'links': 'https://github.com/Aryanmishra299/Pizza_Sales_SQL'}],
 [{'links': 'https://github.com/Aryanmishra299/Guessing-the-number-'}],
 [{'links': 'https://github.com/Aryanmishra299/Guessing-the-number-'}],
 [{'links': 'https://github.com/Aryanmishra299/Guessing-the-number-'}],
 [{'links': 'https://github.com/Aryanmishra299/Guessing-the-number-'}],
 [{'links': 'https://github.com/Aryanmishra299/Guessing-the-number-'}],
 [{'links': 'https://github.com/Aryanmishra299/Guessing-the-number-'}],
 [{'links': 'https://github.com/Aryanmishra299/Snake-Water-Gun-game'}],
 [{'links': 'https://github.com/Aryanmishra299/Snake-Water-Gun-game'}],
 [{'links': 'https://github.com/Aryanmishra299/Guessing-the-number-'}],
 [{'links': 'https://github.com/Aryanmishra299/Snake-Water-Gun-game'}]]

In [87]:
prompt_email = PromptTemplate.from_template(
    """
    ### JOB DESCRIPTION:

    {job_description}

    ### Instruction :
    You are Aryan Mishra, an AI engineer having a experience of 2+year in,
    1. OCR extraction services
    2. Problem-Solving,
    providing the best solution in the definite time of work, effectively , efficient and completing a task in given time. GOt acivement award  for the latest project by client by completing the complexity of the program before deadline.
    Your job is to write a cold mail to the client regarding the job mentioned above in fullfilling the need
    Also add the most relevant from the following projects to showcase the portfolio :{links_list} and how you are the best fit for this role.
    Remember you are Aryan AI ENGINEER.
    ### Email (NO PREAMBLE):

    """
)


chain_email = prompt_email | llm
res = chain_email.invoke({'job_description': str(job), 'links_list': links})
print(res.content)

Subject: Application for Machine Learning Engineer II Role at Amazon

Dear Hiring Manager,

I am excited to apply for the Machine Learning Engineer II role at Amazon, as advertised. With over 2 years of experience in AI engineering, I am confident that my skills and expertise make me an ideal fit for this position.

As a seasoned AI engineer, I have a strong background in problem-solving, having worked on various projects that require efficient and effective solutions. My experience in OCR extraction services has also equipped me with the skills to work with large datasets and develop innovative solutions. I am proud to mention that my latest project received an achievement award from the client for completing the complexity of the program before the deadline.

I am particularly drawn to this role at Amazon because of the opportunity to work with the largest online retail search application in the world. My skills in machine learning approaches and algorithms, as well as my experience 